# Doping and Gap at the Dirac Point of Graphene on SiO2

> **Yong-Ju Kang, Joongoo Kang, and K. J. Chang**, "Electronic structure of graphene and doping
> effect on SiO2", Physical Review B 78, 115404 (2008).
> [DOI:10.1103/PhysRevB.78.115404](https://doi.org/10.1103/PhysRevB.78.115404)

Computes the band structure of graphene on O-terminated α-quartz(0001), in the manuscript's
metastable registry of Sec. III built in the [structure notebook](interface_2d_3d_graphene_silicon_dioxide.ipynb),
and of a free-standing reference of the same strained film, then reads the Dirac point's position
relative to the Fermi level and the gap at K, and compares both with Kang et al. (2008) Sec. III
and Fig. 3(a).

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](interface_2d_3d_graphene_silicon_dioxide.ipynb), which saves them to the `uploads` folder under the names used in cell 1.2 below.
1. Set the material names and parameters in cells 1.2-1.4, or use the defaults; `RELAX = True` relaxes the interface once (fixed-cell, whole slab) and saves it as `<name> relaxed` for reuse by this and other notebooks.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results; the last section prints the comparison with the manuscript.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the materials by name from the `uploads` folder, print their provenance and save them to the platform.
1. Configure the shared DFT model and k-grid.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Relax the interface if `RELAX`, reusing a saved relaxed structure when one already exists.
1. Configure the Band Structure workflow and run one job per material, re-using a job that already ran under the same workflow name.
1. Retrieve the band structures, the Fermi level and the Dirac point's position and gap at K.
1. Compare with Kang et al. (2008).


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")

### 1.2. Material names

In [ ]:
# Names saved by interface_2d_3d_graphene_silicon_dioxide.ipynb.
INTERFACE_NAME = "Graphene 2x2 on SiO2, metastable registry, H-passivated"
REFERENCE_NAME = "Graphene 2x2 on SiO2, free-standing reference"

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
BAND_STRUCTURE_WORKFLOW_SEARCH_TERM = "band_structure.json"
MY_WORKFLOW_NAME = "Band Structure"
APPLICATION_NAME = "espresso"

# NOTE: False reads the band structure as built; True relaxes the interface once (fixed cell, whole
# slab -- made has no per-atom constraint wired to the platform workflow, so the back cannot be
# frozen separately) before the band structure. Kang et al.'s 0.13 eV gap comes from the relaxed O
# positions, so RELAX = True is the regime that can reproduce it; it is also the slowest Group B job.
RELAX = False

CLUSTER_NAME = "001"  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 16  # queue OR on cluster-001 allows at most 16 cores per node
TIME_LIMIT = "24:00:00"  # covers the optional relaxation of this 72-atom cell

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"  # Kang et al. 2008 use LDA
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the only LDA family the platform publishes for Si, O, C and H
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff

KPOINT_DENSITY = 4  # gives 6x6x1 on the measured 21.05 Å² interface cell, Kang et al. Sec. II
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY}"

SCF_UNIT = "pw_scf"
BANDS_UNIT = "pw_bands"
RELAX_UNIT = "pw_relax"
# The interface carries 369 valence electrons (odd); Gaussian smearing handles the fractional
# occupation the same way every other doped or metallic job in this corpus does (D6).
SMEARING_SETTINGS = {"degauss": 0.01}  # Ry
RELAXATION_SETTINGS = {"forc_conv_thr": 1.17e-3, "nstep": 100}  # 0.03 eV/Å, Kang et al. 2008 Sec. II
# Names the relaxation job; the relaxed structure itself is found by content hash.
RELAX_TAG = f"{MODEL_TAG} f{RELAXATION_SETTINGS['forc_conv_thr']}"

KPATH_STEPS = 20
KPATH = [
    {"point": "K", "steps": KPATH_STEPS},
    {"point": "Γ", "steps": KPATH_STEPS},
    {"point": "M", "steps": KPATH_STEPS},
    {"point": "K", "steps": 1},
]

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder and print provenance

The structures, their registry and the back passivation all come from the structure notebook; this one only loads them by name.

In [ ]:
from collections import Counter
from mat3ra.notebooks_utils.core.entity.material.api import load_material

GBRV_VALENCE = {"Si": 4, "O": 6, "C": 4, "H": 1}

materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in (INTERFACE_NAME, REFERENCE_NAME)}
interface, reference = materials_by_name[INTERFACE_NAME], materials_by_name[REFERENCE_NAME]
for name, material in materials_by_name.items():
    composition = Counter(material.basis.elements.values)
    electron_count = sum(GBRV_VALENCE[element] * count for element, count in composition.items())
    print(f"{name}: {dict(composition)}, {material.basis.number_of_atoms} atoms, "
          f"gamma = {material.lattice.gamma:.3f}°, {electron_count} valence electrons "
          f"({'odd' if electron_count % 2 else 'even'})")

### 3.2. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_interface = get_or_create_material(client, interface, ACCOUNT_ID)
saved_reference = get_or_create_material(client, reference, ACCOUNT_ID)

## 4. Configure the shared DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. k-grid per material

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

kgrid = {
    INTERFACE_NAME: kgrid_from_density(interface, KPOINT_DENSITY, periodic_dims=(0, 1)),
    REFERENCE_NAME: kgrid_from_density(reference, KPOINT_DENSITY, periodic_dims=(0, 1)),
}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Relax the interface (optional)

Runs only if `RELAX`: finds any relaxed version of this structure already on the account, regardless of who relaxed it or with what, before spending any compute on a new one. The band structure is then read off that geometry. Kang et al.'s 0.13 eV gap comes from the second surface O relaxing toward a neighbouring C atom, so an unrelaxed run is the as-built geometry rather than a rougher version of the relaxed one.

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.core.entity.material.api import find_relaxed_material, get_final_structure_for_job
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

relaxed_interface = None
if RELAX:
    relax_workflow_name = f"Fixed-cell Relaxation {INTERFACE_NAME} {RELAX_TAG}"
    relaxed_interface = find_relaxed_material(client, interface, ACCOUNT_ID)
    if relaxed_interface is not None:
        print(f"♻️  Relaxed interface: {relaxed_interface.name} ({relaxed_interface.id})")
    else:
        relax_workflow = Workflow.create(
            WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(RELAX_WORKFLOW_SEARCH_TERM)
        )
        relax_workflow.name = relax_workflow_name
        relax_workflow.subworkflows[0].model = model
        apply_planewave_cutoffs(relax_workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
        apply_scf_kgrid(relax_workflow, kgrid[INTERFACE_NAME], material=interface, unit_name=RELAX_UNIT)
        patch_workflow_qe_input(relax_workflow, {"system": SMEARING_SETTINGS}, [RELAX_UNIT])
        patch_workflow_qe_input(relax_workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])

        relax_job = find_job_for_material(
            client, saved_interface["_id"], relax_workflow_name, ACCOUNT_ID,
            statuses=("submitted", "queued", "active", "finished"),
        )
        if relax_job is None:
            relax_job = create_job(
                api_client=client, materials=[saved_interface], workflow=relax_workflow,
                project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
                prefix=f"{relax_workflow_name} {timestamp}",
            )
            submit_jobs(client.jobs, [relax_job["_id"]])
        await wait_for_jobs_to_finish_async(client.jobs, [relax_job["_id"]], poll_interval=POLL_INTERVAL)

        relaxed_material = get_final_structure_for_job(client, relax_job["_id"])
        client.materials.update(relaxed_material.id, {"name": f"{INTERFACE_NAME} relaxed"})
        relaxed_interface = Material.create(client.materials.get(relaxed_material.id))
        print(f"✅ Relaxed interface: {relaxed_interface.name} ({relaxed_interface.id})")

        total_force = get_properties_for_job(client, relax_job["_id"], "total_force")[0]
        print(f"Residual force after relaxation (norm over all atoms): "
              f"{total_force['value']:.4f} {total_force['units']}")

interface_for_jobs = relaxed_interface if RELAX else interface
interface_for_jobs.lattice.type = "HEX"  # the symbolic K-Γ-M-K path needs a named lattice type
saved_interface_for_jobs = get_or_create_material(client, interface_for_jobs, ACCOUNT_ID)

## 7. Band structure jobs
### 7.1. Configure one workflow per material

In [ ]:
from mat3ra.wode.context.providers import PointsPathDataProvider
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from copy import deepcopy

materials_for_jobs = {INTERFACE_NAME: (interface_for_jobs, saved_interface_for_jobs), REFERENCE_NAME: (reference, saved_reference)}
band_structure_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    BAND_STRUCTURE_WORKFLOW_SEARCH_TERM
)

workflows = {}
for name, (material, _) in materials_for_jobs.items():
    workflow = Workflow.create(deepcopy(band_structure_workflow_config))
    workflow.name = f"{MY_WORKFLOW_NAME} {name} {MODEL_TAG}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=SCF_UNIT)
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=BANDS_UNIT)
    apply_scf_kgrid(workflow, kgrid[name], material=material)
    patch_workflow_qe_input(workflow, {"system": SMEARING_SETTINGS}, [SCF_UNIT, BANDS_UNIT])

    bands_subworkflow = workflow.subworkflows[0]
    bands_unit = bands_subworkflow.get_unit_by_name(name=BANDS_UNIT)
    bands_unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data())
    bands_subworkflow.set_unit(bands_unit)

    workflows[name] = workflow
    print(workflow.name)

visualize_workflow(workflows[INTERFACE_NAME])

### 7.2. Run the jobs one at a time

A job is found by its material and its workflow name, which carries the model tag, and re-used if it exists; otherwise it is created and submitted. The jobs run one after another.

In [ ]:
job_ids = {}
for name, workflow in workflows.items():
    _, saved_material = materials_for_jobs[name]
    job = find_job_for_material(
        client, saved_material["_id"], workflow.name, ACCOUNT_ID,
        statuses=("submitted", "queued", "active", "finished"),
    )
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ {name}: submitted job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing job {job['_id']}")
    await wait_for_jobs_to_finish_async(client.jobs, [job["_id"]], poll_interval=POLL_INTERVAL)
    job_ids[name] = job["_id"]

## 8. Retrieve the results
### 8.1. Band structures

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

band_structures = {}
for name, job_id in job_ids.items():
    material, _ = materials_for_jobs[name]
    band_structures[name] = get_properties_for_job(client, job_id, property_name="band_structure")
    visualize_properties(band_structures[name], title=f"Band Structure: {name}",
                         extra_config={"material": material.to_dict()})

### 8.2. Dirac point and gap at K

`KPATH` puts K first (see 1.4), so the first point of the band structure's path is K. At that point, the Dirac point is the midpoint between the band immediately below and the band immediately above the Fermi level, and the gap is their difference -- read directly, with no assumption about which band index that is.

In [ ]:
import numpy as np

K_INDEX = 0  # KPATH's first point is K

dirac_energies, gaps = {}, {}
for name, band_structure in band_structures.items():
    bands = np.array(band_structure[0]["yDataSeries"])
    fermi_energy = band_structure[0]["fermiEnergy"]
    energies_at_k = bands[:, K_INDEX]
    e_above = energies_at_k[energies_at_k > fermi_energy].min()
    e_below = energies_at_k[energies_at_k <= fermi_energy].max()
    dirac_energies[name] = (e_above + e_below) / 2 - fermi_energy
    gaps[name] = e_above - e_below
    print(f"{name}: E_F = {fermi_energy:.4f} eV, E_D - E_F = {dirac_energies[name]:+.4f} eV, "
          f"gap at K = {1000 * gaps[name]:.1f} meV")

## 9. Compare with Kang et al. (2008)

Settings that differ from the manuscript: GBRV ultrasoft LDA pseudopotentials against VASP at 396 eV; the standata quartz cell is +2.3% / +2.0% (a/c) off the manuscript's own LDA-relaxed cell; the substrate is 21 Si planes where the manuscript's "14 bilayers" is not reproduced exactly (D2); the back carries one dangling bond after passivation, not zero (D3).

In [ ]:
KANG_DIRAC_MINUS_FERMI = 1.2  # eV, Fig. 3(a) read-off
KANG_GAP = 0.13  # eV, Sec. III text

regime = "relaxed interface" if RELAX else "unrelaxed SCF"
print(f"Regime: {regime}")
print(f"Doping sign: {'p-type (Dirac point above E_F)' if dirac_energies[INTERFACE_NAME] > 0 else 'n-type (Dirac point below E_F)'}")
print(f"{'E_D - E_F (Kang et al., 2008):':<32}{KANG_DIRAC_MINUS_FERMI:+.3f} eV")
print(f"{'E_D - E_F (this notebook):':<32}{dirac_energies[INTERFACE_NAME]:+.3f} eV "
      f"({100 * (dirac_energies[INTERFACE_NAME] - KANG_DIRAC_MINUS_FERMI) / KANG_DIRAC_MINUS_FERMI:+.1f} % deviation)")
print(f"{'Gap at K (Kang et al., 2008):':<32}{KANG_GAP:.3f} eV")
print(f"{'Gap at K (this notebook):':<32}{gaps[INTERFACE_NAME]:.3f} eV "
      f"({100 * (gaps[INTERFACE_NAME] - KANG_GAP) / KANG_GAP:+.1f} % deviation)")
print(f"{'Gap at K (free-standing reference):':<32}{gaps[REFERENCE_NAME]:.3f} eV, "
      f"E_D - E_F = {dirac_energies[REFERENCE_NAME]:+.3f} eV")

## References

[1] Y.-J. Kang, J. Kang and K. J. Chang, "Electronic structure of graphene and doping effect on SiO2", Phys. Rev. B 78, 115404 (2008). https://doi.org/10.1103/PhysRevB.78.115404

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053